# 02 - Containerisation
---

### **Introduction**
When code runs, it depends on the following:
- The programming language version (e.g. Python 3.10)
- The packages and their versions (e.g pandas 2.2.2)      
- The operating system and its version (e.g. linux 24.04.1)            
- Hardware drivers and their versions (e.g. NVIDIA CUDA 12.2)

Differences between the above can lead to different behaviour and outcomes when the code is run. This is why you get the classic "It works on my machine" problem among developers. This issue becomes even more acute when trying to deploy code to production as any differences can lead to the code not performing in your production environment which is a serious problem. Issues can lead to lost revenue, reputational damage and high costs to fix. 

**Containerisation** is a solution to this issue. It works by packaging up not just the code itself, but also all the environmental elements. The code is then run inside this specially created environment called a container. There are several benefits of doing this:
- The exact desired environment is set, ensuring the code runs as expected
- The container can be tested by running it locally before deploying it to production; this gives greater confidence when deploying
- The code can be deployed without changes being needed

### **Docker**
Docker is an open source tool used to build, manage and run containers. A **Docker Image** is essentially a recipe which tells the **Docker Container** how the environment should be specified and what code should be executed in that environment. Docker images are highly customisable, allowing for a wide variety of operating systems, programming languages, package managers etc. to be used. 

Docker images are specified by a `Dockerfile` which typically sits in the root of your project. For most python ML use cases they will typically involve the following steps:
- Set a base image
- Setup a virtual environment
- Install dependencies
- Copy relevant files
- Run code

Note that a base image is a prebuilt Docker image and typically contains an operating system plus an installation of a coding language such as python. We usually base our Docker image on one of these base images since it is easier than trying to install an operating system and python ourselves. 

### **Example - Simple Docker Image**
A simple example of a Dockerfile is shown below. It does the following:
- Uses `python:3:11-slim` as a base
- Creates a virtual environment
- Installs requirements from a `requirements.txt` file
- Copies some files accross into the container
- Sets an environment variable
- Runs a `main.py` script

```docker
# Use official Python runtime as base image
FROM python:3.11-slim

# Set working directory inside container
WORKDIR /app

# Create and activate virtual environment
RUN python -m venv .venv
ENV PATH="/app/.venv/bin:$PATH"

# Copy and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code into container
COPY utils utils
COPY main.py .

# Set environment variable
ENV APP_ENV=production

# Define default command to run application
CMD ["python", "main.py"]
```

### **Example - Multi-stage Dokcer Image**
It is possible to write a Dockerfile in such a way so as to create two different images. The example below demonstrates a minimal example for how this could work. It builds two different images, one for a fast-api endpoint and one for running the `main.py` script.

```docker
# ---------------------
# Base Stage (shared)
# ---------------------
FROM python:3.11-slim AS base

WORKDIR /app

# Create and activate virtual environment
RUN python -m venv .venv
ENV PATH="/app/.venv/bin:$PATH"

# Copy shared utils dir
COPY utils utils


# ---------------------
# FastAPI Final
# ---------------------
FROM base AS fastapi

# Copy code into container
COPY app.py .

# Install FastAPI-specific dependencies
COPY fast_api_requirements.txt .
RUN pip install --no-cache-dir -r fast_api_requirements.txt

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]


# ---------------------
# main.py Final
# ---------------------
FROM base AS main

# Copy code into container
COPY main.py .

# Install main-specific dependencies
COPY main_requirements.txt .
RUN pip install --no-cache-dir -r main_requirements.txt

CMD ["python", "main.py"]
```

### **Docker Terminal Commands**
Docker images need to be built via:
```
docker build .
```

They can be tagged via:
```
docker build -t <tag>:<version> . # note <version> can be latest
```

The specific stage of a multi-stage image can be selected via:
```
docker build --target <target>
```

Docker containers are run via:
```
docker run <image>
```

The container can be named via:
```
docker run --name <name> <image>
```

Environment variables are passed via
```
docker run -e <ENV_VAR>=<value> <image>
```

You can set the container to be automatically removed after it stops via:
```
docker run --rm <image>
```

Images are listed via:
```
docker images
```

Running containers are listed via
```
docker ps
```

All containers (including stopped) are listed via:
```
docker ps -a
```

A container is stopped via:
```
docker stop <container_name_or_id>
```

A container is removed via:
```
docker rm <container_name_or_id>
```

An image is removed via:
```
docker rmi <image>
```

A container's logs are viewed via:
```
docker logs <container_name_or_id>
```

The following command removes stopped containers, unused networks, dangling images (untagged images) and the build cache
```
docker system prune # optionallt add the flag -f to skip confirmation and -a to remove all unused images
```

See how much space Docker is using via:
```
docker system df
```
























### **Container Registries**
Many cloud computing providers (e.g. Azure, AWS, GCP) offer a version of a container registry which is a service for managing Docker images. They allow you to manage and version docker images. Other cloud services can then be pointed at a particular image to run it.

For instance, you can upload an image to the Azure Container Registry (ACR) via:
```
docker push <reigstry>.azurecr.io/<image>:<version>
```

Note the above assumes you have an ACR setup called \<registry\> and that you have already authed with azure from the command line (e.g. `az login` and `az acr login --name <acr_name>`)

List the images in the ACR via:
```
az acr repository list --name <registry> --output table
```
